In [5]:
# --- 1. Import Libraries ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- 2. Load Dataset ---
df = pd.read_csv("cleaned_croma_laptops.csv")

# --- 3. Features & Target ---
X = df.drop("Price", axis=1)   # predictors
y = df["Price"]                # target variable

# --- 4. Train-Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- 5. Identify Categorical & Numeric Features ---
categorical_features = X.select_dtypes(include=["object"]).columns
numeric_features = X.select_dtypes(include=[np.number]).columns

# --- 6. Preprocessing ---
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# --- 7. Build Lasso Model ---
lasso_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Lasso(alpha=0.001, max_iter=10000, random_state=42))
])

# --- 8. Train Model ---
lasso_model.fit(X_train, y_train)

# --- 9. Predictions ---
y_pred = lasso_model.predict(X_test)

# --- 10. Evaluation Metrics ---
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Lasso Regression Results")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)


Lasso Regression Results
MAE : 24929.24192319026
RMSE: 33466.506456812975
R²  : 0.7373094528365473


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:658: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 76441448918.57365, tolerance: 170273229.87219578
  model = cd_fast.sparse_enet_coordinate_descent(


In [6]:
import pickle

# Save the trained Ridge model
with open("lasso_model.pkl", "wb") as file:
    pickle.dump(lasso_model, file)